# 统一30近邻评估

固定原数据、权重与外层划分，统一使用真实Subclass。先用10近邻回归原总分及类型中位数，再计算30近邻miLISI。全量和消融复用冻结PCA坐标，并重算30近邻UMAP；九子集按原训练参照重算ANOVA50000/PCA50。A1/A2逐值复用已核验结果，不重训或重新选模。旧checkpoint的原内部选择协议保留历史记录，未来训练入口统一30近邻。

In [ ]:
from pathlib import Path
from datetime import datetime,timezone,timedelta
import json,hashlib,shutil,sys,os,time,gc
import nbformat,yaml
import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.decomposition import PCA
from threadpoolctl import threadpool_limits

PROJECT=Path('/home/sccasimp/program/xb/simscDNAm/releases/v1_knn30_20261111')
ROOT=Path('/home/sccasimp/data/xb/outputs/simscDNAm')
RUN=ROOT/'simscDNAm_knn30_20261111'
MODE='subset'
DATASET='liu2023/GSE213221'
directory=RUN/(DATASET if MODE!='ablation' else 'ablation/liu2023/full')
directory.mkdir(parents=True,exist_ok=True)
sys.path.insert(0,str(PROJECT))
os.chdir(PROJECT)
from evaluate.metrics import milisi
from evaluate.preprocessing import (natural_reference_statistics,select_features,imputed_features,
                                    read_labels,validate_region_axis)
assert Path(sys.prefix).name.lower()=='simscdnam'
START=time.monotonic()

def write_json(path,value):
    path.write_text(json.dumps(value,ensure_ascii=False,indent=2),encoding='utf-8')

def progress(stage,**extra):
    value={'阶段':stage,'数据集':DATASET,'模式':MODE,
           '北京时间':datetime.now(timezone(timedelta(hours=8))).isoformat(),
           '已用秒数':round(time.monotonic()-START,1),**extra}
    write_json(directory/'progress.json',value)
    print(value,flush=True)

def memory_data(path):
    item=ad.read_h5ad(path)
    if sp.issparse(item.X):item.X=item.X.toarray()
    return item

progress('核对原始输入')


In [ ]:
if MODE=='subset':
    source=ROOT/'simscDNAm_subset_comparison_20261111'/DATASET
    prior_verification=json.loads((source/'verification.json').read_text())
    assert prior_verification['状态']=='通过'
    input_records=dict(prior_verification['输入'])
    prepared=ROOT/'six_method_subsets_20261024'/DATASET/'expanded/legacy/methods'
    paths={'训练参照':prepared/'Truth/data.h5ad','真实数据':prepared/'HeldoutReal/data.h5ad',
           'simscDNAm':source/'simscDNAm/data.h5ad',
           **{name:prepared/name/'data.h5ad' for name in ['FittedMarginal','simATAC','simCAS']}}
    methods=['simscDNAm','FittedMarginal','simATAC','simCAS']
    old_scores_path=source/'scores.csv'
    for p in paths.values():input_records.setdefault(str(p),[p.stat().st_size,p.stat().st_mtime_ns])
    for p,before in input_records.items():
        stat=Path(p).stat()
        assert [stat.st_size,stat.st_mtime_ns]==before,p
    progress('训练参照ANOVA与PCA')
    representations={};labels={};ids={}
    with threadpool_limits(limits=8):
        training=memory_data(paths['训练参照'])
        natural=training.n_vars-1000
        fill,anova=natural_reference_statistics(training,np.arange(training.n_obs),natural)
        selected=select_features(anova,50000)
        components=min(50,training.n_obs-1,len(selected))
        pca=PCA(n_components=components,svd_solver='randomized',random_state=42)
        representations['训练参照']=pca.fit_transform(imputed_features(training,np.arange(training.n_obs),selected,fill))
        labels['训练参照']=read_labels(training,np.arange(training.n_obs))
        ids['训练参照']=training.obs_names.to_numpy()
        pd.DataFrame({'原region索引':selected,'原region':training.var_names[selected],
                      'ANOVA':anova[selected]}).to_csv(directory/'A3_selected_regions.csv',index=False)
        for name in ['真实数据',*methods]:
            progress('投影到同一PCA空间',方法=name)
            item=memory_data(paths[name])
            validate_region_axis(training,item)
            rows=np.arange(item.n_obs)
            labels[name]=read_labels(item,rows)
            ids[name]=item.obs_names.to_numpy()
            representations[name]=pca.transform(imputed_features(item,rows,selected,fill))
            del item
        del training
        gc.collect()
    tables=[]
    for name,rep in representations.items():
        frame=pd.DataFrame(rep,columns=[f'PC{i+1}' for i in range(components)])
        frame.insert(0,'细胞类型',labels[name]);frame.insert(0,'细胞ID',ids[name]);frame.insert(0,'数据集',name)
        tables.append(frame)
    coordinates=pd.concat(tables,ignore_index=True)
    coordinates.to_csv(directory/'A3_PCA_coordinates.csv',index=False)
    training_count=prior_verification['训练细胞数']
    natural_count=prior_verification['自然区域数']
    config={'milisi_neighbors':30,'umap_neighbors':30,'a3_top_regions':50000,'pca_components':50,
            'seed':42,'label_key':'Subclass','输入':{k:str(v) for k,v in paths.items()}}
else:
    source=ROOT/('simscDNAm_release_20261110/evaluation' if MODE=='full' else 'simscDNAm_full_ablation_20261111/evaluation')
    prior_verification=json.loads((source/'run_summary.json').read_text())
    assert prior_verification['状态']=='完成'
    input_records={item['路径']:[item['字节数'],item['修改时间纳秒']] for item in prior_verification['输入文件']}
    checkpoint=ROOT/'CVAE_training_mask_liu2023_full_20261107_1450/liu2023/full/CVAE_training_mask/model.pt'
    input_records[str(checkpoint)]=[checkpoint.stat().st_size,checkpoint.stat().st_mtime_ns]
    pc_path=source/'A3_PCA_coordinates.csv'
    input_records[str(pc_path)]=[pc_path.stat().st_size,pc_path.stat().st_mtime_ns]
    coordinates=pd.read_csv(pc_path)
    pc_columns=[c for c in coordinates if c.startswith('PC')]
    components=len(pc_columns)
    selected=pd.read_csv(source/'A3_selected_regions.csv')['原region索引'].to_numpy()
    representations={};labels={};ids={}
    for name,frame in coordinates.groupby('数据集',sort=False):
        representations[name]=frame[pc_columns].to_numpy()
        labels[name]=frame['细胞类型'].astype(str).to_numpy()
        ids[name]=frame['细胞ID'].to_numpy()
    shutil.copy2(pc_path,directory/'A3_PCA_coordinates.csv')
    shutil.copy2(source/'A3_selected_regions.csv',directory/'A3_selected_regions.csv')
    methods=prior_verification['方法']
    old_scores_path=source/'A1_A2_A3_summary.csv'
    training_count=prior_verification['真实参照细胞数']
    natural_count=prior_verification['原始区间数']
    config=dict(prior_verification['配置'])
    config.update({'milisi_neighbors':30,'output_dir':str(directory),
                   'a1_regions_csv':str(PROJECT/'evaluate/a1_marker_regions_liu2023.csv')})
    (directory/'figures').mkdir(exist_ok=True)
    for stem in ['A1_marginal_regions','A1_region_by_type','A2_duplicate_Pearson']:
        for suffix in ['png','pdf']:shutil.copy2(source/'figures'/f'{stem}.{suffix}',directory/'figures'/f'{stem}.{suffix}')
    for name in ['input_diagnostics.csv','A1_type_details.csv','A2_duplicate_correlations.csv',
                 'A1_plot_regions.csv','A1_plot_histograms.csv','A1_type_histograms.csv']:
        if (source/name).exists():shutil.copy2(source/name,directory/name)

old_scores=pd.read_csv(old_scores_path).set_index('方法').loc[methods]
assert old_scores['近邻数'].eq(10).all()
old_types=pd.read_csv(source/'A3_type_details.csv')
for path in [old_scores_path,source/'A3_type_details.csv']:
    input_records[str(path)]=[path.stat().st_size,path.stat().st_mtime_ns]
CONFIG_PATH=directory/'config.yaml'
CONFIG_PATH.write_text(yaml.safe_dump(config,allow_unicode=True,sort_keys=False),encoding='utf-8')
progress('PCA空间准备完成',训练细胞数=training_count,留出细胞数=len(ids['真实数据']))


In [ ]:
records=[];cell_tables=[];type_tables=[];regression=[]
with threadpool_limits(limits=8):
    for name in methods:
        progress('复核10近邻并计算30近邻',方法=name)
        score10,cells10,groups10=milisi(representations['真实数据'],labels['真实数据'],
            representations[name],labels[name],n_neighbors=10)
        np.testing.assert_allclose(score10,old_scores.loc[name,'miLISI'],rtol=1e-7,atol=1e-9)
        expected=old_types[old_types['方法']==name].set_index('细胞类型')['miLISI中位数'].sort_index()
        actual=groups10.set_index('细胞类型')['miLISI中位数'].sort_index()
        assert actual.index.equals(expected.index)
        np.testing.assert_allclose(actual,expected,rtol=1e-7,atol=1e-9)
        score,cells,groups=milisi(representations['真实数据'],labels['真实数据'],
            representations[name],labels[name],n_neighbors=30)
        record=old_scores.loc[name].to_dict()
        record.update({'方法':name,'miLISI':score,'近邻数':30,'入选区域数':len(selected),'PCA维数':components})
        records.append(record)
        cells.insert(0,'细胞ID',np.r_[ids['真实数据'],ids[name]])
        cells.insert(0,'方法',name);groups.insert(0,'方法',name)
        cell_tables.append(cells);type_tables.append(groups)
        regression.append({'方法':name,'旧10近邻':old_scores.loc[name,'miLISI'],'复算10近邻':score10,
                           '新30近邻':score,'10近邻回归最大类型误差':float(np.max(np.abs(actual-expected)))})
summary=pd.DataFrame(records)
columns=['加权 1-WD','加权 MSE','加权 Pearson','A2_Pearson均值','miLISI']
assert np.isfinite(summary[columns].to_numpy()).all()
np.testing.assert_array_equal(summary.set_index('方法').loc[methods,columns[:4]].to_numpy(),old_scores[columns[:4]].to_numpy())
summary.to_csv(directory/'A1_A2_A3_summary.csv',index=False)
summary.to_csv(directory/'scores.csv',index=False)
summary[['方法','miLISI','近邻数','入选区域数','PCA维数']].to_csv(directory/'A3_miLISI.csv',index=False)
pd.concat(cell_tables,ignore_index=True).to_csv(directory/'A3_miLISI_cells.csv',index=False)
pd.concat(type_tables,ignore_index=True).to_csv(directory/'A3_type_details.csv',index=False)
pd.DataFrame(regression).to_csv(directory/'k10_regression_k30.csv',index=False)
display(summary)
progress('30近邻得分完成')


In [ ]:
if MODE!='subset':
    evaluation=nbformat.read(PROJECT/'evaluate/A1A2A3.ipynb',as_version=4)
    exec(evaluation.cells[3].source,globals())
    exec(evaluation.cells[5].source,globals())
    if MODE=='ablation':
        PLOT_COLORS.update({'CVAE_raw':'#7A8BA5','CVAE_mask':'#3B73B9','CVAE_align':'#D57A24','simscDNAm':'#269D91'})
        original_plot_name=plot_name
        def plot_name(name):
            return {'CVAE_raw':'CVAE','CVAE_mask':'CVAE + mask','CVAE_align':'CVAE + alignment',
                    'simscDNAm':'simscDNAm (full)'}.get(name,original_plot_name(name))
    joint_tables=[]
    with threadpool_limits(limits=8):
        for name in methods:
            progress('重算30近邻UMAP',方法=name)
            combined=np.vstack([representations['真实数据'],representations[name]])
            xy=umap.UMAP(n_neighbors=30,random_state=2022,transform_seed=2022,n_jobs=1).fit_transform(combined)
            joint_tables.append(pd.DataFrame({'方法':name,
                '来源':np.r_[np.repeat('真实',len(ids['真实数据'])),np.repeat('合成',len(ids[name]))],
                '细胞ID':np.r_[ids['真实数据'],ids[name]],'UMAP1':xy[:,0],'UMAP2':xy[:,1]}))
    joint=pd.concat(joint_tables,ignore_index=True)
    joint.to_csv(directory/'A3_joint_coordinates.csv',index=False)
    plot_joint_umap(joint,summary,methods,directory,42)
    assert len(list((directory/'figures').glob('*')))==8

for path,before in input_records.items():
    stat=Path(path).stat()
    assert [stat.st_size,stat.st_mtime_ns]==before,path
verification={'状态':'通过','数据集':DATASET,'模式':MODE,'近邻数':30,'UMAP近邻数':30 if MODE!='subset' else None,
    '训练细胞数':training_count,'留出细胞数':len(ids['真实数据']),'自然区域数':natural_count,
    '入选区域数':len(selected),'PCA维数':components,'原10近邻总分及类型中位数回归通过':True,
    'A1A2逐值保持':True,'所有五项指标有限':True,'未重新训练或选模':True,
    '权重与合成矩阵状态未变':True,'PCA模式':'冻结坐标' if MODE!='subset' else '相同训练参照重算并回归',
    '旧得分来源':str(old_scores_path),'旧得分SHA256':hashlib.sha256(old_scores_path.read_bytes()).hexdigest(),
    '输入文件状态':input_records,'指标代码SHA256':hashlib.sha256((PROJECT/'evaluate/metrics.py').read_bytes()).hexdigest()}
write_json(directory/'verification.json',verification)
write_json(directory/'run_summary.json',{'状态':'完成','配置':config,'方法':methods,'核验':verification})
progress('完成',核验='通过')
